# P4 — Inventory builder and visualization

This stage sits at the end of the pipeline. Each earlier stage contributes one field of a detection record:

1. **P1** finds a brick and returns its bounding box.
2. **P2** classifies the crop and returns a part id (`predict` in `P2/classifier.py`).
3. **P3** classifies the crop and returns a color id.

P4 groups those records by part id and color id, counts them, and keeps the boxes so the same pieces can be highlighted on the input photo.

P1 and P3 do not have a real output format in the repo yet, so the cells below build **mock records** in the shape those stages will pass in. Part ids are real ids from `P2/classes_50.csv`.

## Load the inventory code

In [ ]:
import sys
from pathlib import Path

from PIL import Image, ImageDraw
from IPython.display import display

# add P4 to the path so import inventory works from this folder or the repo root
search_dirs = [Path.cwd(), Path.cwd() / "P4", Path.cwd().parent]
for search_dir in search_dirs:
    inventory_file = search_dir / "inventory.py"
    if inventory_file.exists():
        sys.path.insert(0, str(search_dir))
        break

from inventory import annotate_image, build_inventory

## Mock photo and mock detections

The rectangles stand in for bricks a detector would find. Each detection is one brick: a box, a part id, and a color id. Two red 2x4 bricks share a part and a color, so they should become one inventory line with count 2. The blue 2x4 has the same part id but a different color id, so it stays on its own line.

In [ ]:
# rough lego colors so the fake photo looks like bricks. keys are P3 color ids
BRICK_COLORS = {
    0: (35, 35, 35),       # black
    1: (0, 85, 191),       # blue
    4: (196, 40, 28),      # red
    14: (245, 205, 47),    # yellow
    15: (245, 245, 245),   # white
}

# names are just for the printout
PART_NAMES = {
    "3001": "2x4 Brick",
    "3003": "2x2 Brick",
    "3023": "1x2 Plate",
    "3710": "1x4 Plate",
}

mock_detections = [
    {"bounding_box": (40, 70, 220, 170), "part_id": "3001", "color_id": 4},
    {"bounding_box": (260, 70, 440, 170), "part_id": "3001", "color_id": 4},
    {"bounding_box": (520, 70, 700, 170), "part_id": "3001", "color_id": 1},
    {"bounding_box": (40, 280, 150, 340), "part_id": "3023", "color_id": 14},
    {"bounding_box": (180, 280, 290, 340), "part_id": "3023", "color_id": 14},
    {"bounding_box": (320, 280, 430, 340), "part_id": "3023", "color_id": 14},
    {"bounding_box": (520, 260, 640, 380), "part_id": "3003", "color_id": 0},
    {"bounding_box": (40, 420, 300, 490), "part_id": "3710", "color_id": 15},
]

# stand-in photo. real pipeline uses the users photo and boxes from P1
scene = Image.new("RGB", (780, 540), (225, 225, 225))
scene_draw = ImageDraw.Draw(scene)
for detection in mock_detections:
    bounding_box = detection["bounding_box"]
    fill_color = BRICK_COLORS[detection["color_id"]]
    scene_draw.rectangle(bounding_box, fill=fill_color, outline=(80, 80, 80), width=2)

print("Mock detections:", len(mock_detections))
display(scene)

## Build the inventory

`build_inventory` groups on `(part_id, color_id)`, counts the bricks in each group, and stores the boxes.

In [ ]:
inventory = build_inventory(mock_detections)

print("part_id  name        color_id  count  boxes")
print("-" * 72)
total_bricks = 0
for group in inventory:
    part_id = group["part_id"]
    part_name = PART_NAMES.get(part_id, "")
    part_name = (part_name + " " * 10)[:10]
    total_bricks = total_bricks + group["count"]
    print(
        part_id,
        " ",
        part_name,
        "   ",
        str(group["color_id"]).rjust(3),
        "     ",
        str(group["count"]).rjust(2),
        "   ",
        group["boxes"],
    )

print()
print(
    str(total_bricks)
    + " bricks grouped into "
    + str(len(inventory))
    + " inventory lines"
)

## Highlight the pieces

Each inventory line gets one outline color. The two red 2x4 bricks share a color. The blue 2x4 is the same part, so it is a different line and a different outline.

In [ ]:
annotated_image = annotate_image(scene, inventory)
display(annotated_image)